## How to run in Colab
1. Download the dataset zip [here](https://github.com/devtlv/Datasets-GEN-AI-Bootcamp/raw/refs/heads/main/Week%203/W3D4%20-%20Mini%20Project/Apple%20Stock%20Prices%20From%201981%20to%202023.zip) and save it in `C:\Users\sivan\Datasets`.
2. Open [Google Colab](https://colab.research.google.com) → **File → Upload notebook** → choose this file.
3. Run the first code cell and, when it asks, choose `Apple Stock Prices From 1981 to 2023.zip` from `C:\Users\sivan\Datasets`.

**Run the cells from top to bottom** (Shift+Enter runs one cell).

In [ ]:
from google.colab import files
# colab tool for uploading files from my computer
uploaded = files.upload()
# opens a window where I choose the zip
!unzip -o "Apple Stock Prices From 1981 to 2023.zip"
# unzips it, the csv I need is "Apple Stock Prices (1981 to 2023).csv"

# Week 3 Day 3 – Mini-project: Advanced Statistical Analysis of Apple Inc. Stock Data

**What I learn here**
- Master statistical analysis of financial data using NumPy and SciPy.
- Learn effective data visualization techniques with Matplotlib for financial trends.
- Apply hypothesis testing to financial datasets for meaningful insights.
- Understand and use advanced statistical techniques in NumPy and SciPy.

In this project I analyze the AAPL (Apple Inc.) daily stock prices from 1981 to 2023: Date, Open, High, Low, Close, Adj Close and Volume.

**Structure**
1. Data Loading and Exploration
2. Data Visualization
3. Statistical Analysis
4. Hypothesis Testing
5. Advanced Statistical Techniques (Bonus)
6. Summary and Insights
7. Reflection

In [ ]:
import numpy as np
# numpy for math and arrays
import pandas as pd
# pandas for the table
import matplotlib.pyplot as plt
# matplotlib for charts
from scipy import stats
# scipy.stats for the hypothesis tests
from scipy import signal
# scipy.signal for the filters in the bonus part

## 1. Data Loading and Exploration
- Load the dataset using Pandas. Check for null values and understand data types.
- Examine the time series properties of the data (frequency, trends).

In [ ]:
df = pd.read_csv('Apple Stock Prices (1981 to 2023).csv')
print(df.shape)
df.head()

In [ ]:
df.info()
# shows the type of every column
print(df.isnull().sum())
# number of empty cells in each column

**What I found:** the file has 10,608 rows and 7 columns, and there are no missing values at all. All price columns are `float64` and Volume is `int64`. The only problem is `Date`: it is read as text and written day/month/year (02/01/1981 is 2 January), so I convert it to a real date in the next step.

In [ ]:
# the dates are written day/month/year, so I tell pandas the format
df['Date'] = pd.to_datetime(df['Date'], format='%d/%m/%Y')
df = df.sort_values('Date').set_index('Date')
# the date becomes the index, so I can slice by date
print(df.index.min().date(), 'to', df.index.max().date())
print('Trading days:', len(df))

In [ ]:
# gap in days between two rows in a row
gaps = df.index.to_series().diff().dt.days
print(gaps.value_counts().head())
# trading days per year (2023 only has January)
per_year = df.groupby(df.index.year).size()
print(per_year.tail())
print('Average trading days per full year:', round(per_year.loc[1981:2022].mean(), 1))

In [ ]:
# yearly average close shows the long term trend
yearly_close = df['Close'].groupby(df.index.year).mean()
print(yearly_close.round(2).iloc[::5])
# every 5th year, to keep it short

**What I found about the time series:**
- The data runs from 1981-01-02 to 2023-01-27, 10,608 trading days.
- The frequency is **trading days**, not calendar days. Most gaps are 1 day (8,317), and the 3-day gaps (1,927) are weekends. The 2 and 4 day gaps are holidays. A full year has about 252 trading days.
- 2023 has only 18 days (January), so I leave it out when I compare full years.
- The trend is a huge long-term rise: the yearly average close went from about 0.11 USD in 1981 to about 141 USD in 2021. The prices are split-adjusted, which is why the old prices look so small. Most of the growth came after 2005.

## 2. Data Visualization
- Plot closing prices and traded volume over time.
- Create a candlestick chart to show high and low prices.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
# two charts, one above the other, with the same date axis
axes[0].plot(df.index, df['Close'])
axes[0].set_title('AAPL Closing Price 1981-2023')
axes[0].set_ylabel('Close (USD)')
axes[1].plot(df.index, df['Volume'], color='gray')
axes[1].set_title('AAPL Traded Volume 1981-2023')
axes[1].set_ylabel('Shares traded')
axes[1].set_xlabel('Date')
plt.tight_layout()
plt.show()

In [ ]:
# the price grew so much that a log scale shows the old years better
plt.figure(figsize=(12, 4))
plt.plot(df.index, df['Close'])
plt.yscale('log')
plt.title('AAPL Closing Price (log scale)')
plt.xlabel('Date')
plt.ylabel('Close (USD, log)')
plt.show()

**What I found:** on the normal scale the price looks flat until about 2005, then rises fast, with the top close of 182.01 USD on 2022-01-03. The log-scale chart shows the early years better: a rise in the 80s, a long fall in the 90s, and steady growth since the 2000s. Volume behaves differently. It peaked around 2000 and 2008 (the biggest day was 2000-09-29, the day of the biggest crash) and has gone down in the last 10 years, even though the price went up.

### Candlestick chart
With 10,000+ days the candles would be too thin to see, so I draw only the last 60 trading days. I build it with plain matplotlib: a thin line from Low to High, and a thick bar from Open to Close (green when the price went up that day, red when it went down).

In [ ]:
last = df.tail(60)
# last 60 trading days
colors = np.where(last['Close'] >= last['Open'], 'green', 'red')
# green day = closed higher than it opened
plt.figure(figsize=(12, 5))
plt.vlines(last.index, last['Low'], last['High'], color=colors, linewidth=1)
# thin line: lowest to highest price of the day
plt.bar(last.index, last['Close'] - last['Open'], bottom=last['Open'], color=colors, width=0.6)
# thick bar: from open to close
plt.title('AAPL Candlestick Chart - last 60 trading days')
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.show()

**What I found:** the window starts on 2022-11-01. Of the 60 days, 28 are green and 32 are red, so slightly more down days. The highest price was 155.45 USD on the first day, and the lowest was 124.17 USD on 2023-01-03. After that the price recovered to about 146 USD at the end of January. The long thin lines show days with a big gap between the high and the low.

## 3. Statistical Analysis
- Compute summary statistics (mean, median, standard deviation) for key columns.
- Analyze closing prices with a moving average.

In [ ]:
pd.options.display.float_format = '{:,.2f}'.format
# shows normal numbers with 2 decimals instead of 1.6e+01
key_cols = ['Open', 'High', 'Low', 'Close', 'Volume']
summary = df[key_cols].agg(['mean', 'median', 'std']).T
# .T flips it so every column is one row
print(summary.round(2))

In [ ]:
# the same stats only for the last full 5 years, because the old prices are tiny
recent = df.loc['2018':'2022']
print(recent[key_cols].agg(['mean', 'median', 'std']).T.round(2))

**What I found:** over the whole period the mean close (16.70 USD) is far above the median (0.49 USD), and the std (35.47) is double the mean. That shows the distribution is very skewed: for more than half of the 42 years the price was under 1 USD, and the few recent years pull the mean up. That is why I also looked at 2018-2022 only. There the mean (98.09) and median (91.63) are close, so it is a more useful summary. Volume tells the opposite story: the average volume in 2018-2022 (about 117 million shares) is much lower than the overall average (about 328 million).

In [ ]:
# moving average = average of the last N closing prices
df['MA50'] = df['Close'].rolling(50).mean()
df['MA200'] = df['Close'].rolling(200).mean()
part = df.loc['2019':]
plt.figure(figsize=(12, 5))
plt.plot(part.index, part['Close'], label='Close', alpha=0.6)
plt.plot(part.index, part['MA50'], label='50-day MA')
plt.plot(part.index, part['MA200'], label='200-day MA')
plt.title('AAPL Close with 50 and 200 day Moving Averages (2019-2023)')
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.legend()
plt.show()

In [ ]:
# golden cross = MA50 goes above MA200, death cross = goes below
above = df['MA50'] > df['MA200']
cross = above.astype(int).diff()
print('Golden crosses since 2019:', list(cross.loc['2019':][cross.loc['2019':] == 1].index.date))
print('Death crosses since 2019:', list(cross.loc['2019':][cross.loc['2019':] == -1].index.date))

**What I found:** the 50-day average follows the price closely, and the 200-day average shows the main trend. From 2019 to the end of 2021 the 50-day line stayed above the 200-day line, which is a strong uptrend. The golden cross on 2019-05-07 started it. In 2022 the 50-day line crossed below on 2022-06-03 (a death cross), which matches the 2022 drop. On 2022-09-28 and 2022-09-30 the two lines crossed twice in 3 days, so a cross is not always a clear signal.

## 4. Hypothesis Testing
- Run a t-test to compare average closing prices across different years.
- Examine the distribution of daily returns and test for normality.

In [ ]:
close_2021 = df.loc['2021', 'Close']
close_2022 = df.loc['2022', 'Close']
print('2021 mean:', round(close_2021.mean(), 2), '| 2022 mean:', round(close_2022.mean(), 2))
# H0: the average close in 2021 and 2022 is the same
t_stat, p_value = stats.ttest_ind(close_2021, close_2022, equal_var=False)
print('t-statistic:', round(t_stat, 2), '| p-value:', p_value)
if p_value < 0.05:
    print('Reject H0: the average close is different')
else:
    print('Fail to reject H0')

In [ ]:
# one-way ANOVA across several recent years at once
years = [2018, 2019, 2020, 2021, 2022]
groups = [df.loc[str(y), 'Close'] for y in years]
f_stat, p_value = stats.f_oneway(*groups)
print('Means:', [round(gr.mean(), 2) for gr in groups])
print('F-statistic:', round(f_stat, 2), '| p-value:', p_value)

**What I found:**
- **t-test 2021 vs 2022:** the mean close was 140.99 USD in 2021 and 154.84 USD in 2022. The t-statistic is -11.19 and the p-value is about 4.7e-26, so I reject H0: the two years have different average closes. I used `equal_var=False` (Welch's t-test) because the spread is not the same each year.
- **ANOVA 2018-2022:** the means go up every year (47.26, 52.06, 95.35, 140.99, 154.84). F = 3184.92 with p about 0, so at least one year is different.
- Note: stock prices from one day to the next are not independent, which the t-test assumes. So the p-values are too optimistic, but the gap between the years is so big that the conclusion does not change.

In [ ]:
# daily return = percent change from yesterday's close
df['Return'] = df['Close'].pct_change()
returns = df['Return'].dropna()
print(returns.describe().round(4))
print('Skew:', round(stats.skew(returns), 2), '| Kurtosis:', round(stats.kurtosis(returns), 2))

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(returns, bins=100, density=True, alpha=0.7, label='Daily returns')
x = np.linspace(-0.2, 0.2, 400)
plt.plot(x, stats.norm.pdf(x, returns.mean(), returns.std()), color='red', label='Normal curve')
# normal curve with the same mean and std, for comparison
plt.xlim(-0.2, 0.2)
plt.title('Distribution of AAPL Daily Returns')
plt.xlabel('Daily return')
plt.ylabel('Density')
plt.legend()
plt.show()

In [ ]:
# shapiro is only accurate up to about 5000 values, so I test a random sample of 5000
sample = returns.sample(5000, random_state=42)
w_stat, p_shapiro = stats.shapiro(sample)
print('Shapiro-Wilk: W =', round(w_stat, 4), '| p-value:', p_shapiro)
# normaltest works on all the returns
k_stat, p_normal = stats.normaltest(returns)
print("D'Agostino normaltest: stat =", round(k_stat, 1), '| p-value:', p_normal)
# H0: the returns are normally distributed

**What I found:**
- The average daily return is about 0.11% and the std is about 2.8%. The worst day was -51.9% (2000-09-29) and the best was +33.2% (1997-08-06).
- The kurtosis is 18.13 (a normal curve has 0), and the skew is -0.38. The histogram shows a much higher peak in the middle and fatter tails than the red normal curve.
- **Shapiro-Wilk** only works well with up to 5000 values, so I tested a random sample of 5000: W = 0.9532, p about 2.7e-37. **normaltest** on all returns: p = 0.0. Both reject H0.
- The daily returns are **not normal**. Very big moves happen much more often than a normal curve predicts, which matters for any risk calculation.

## 5. Advanced Statistical Techniques (Bonus)
### Signal Processing using SciPy
I treat the closing price as a noisy signal and smooth it with two filters, trying different parameters:
- `savgol_filter` (Savitzky-Golay) fits a small polynomial in a moving window.
- `butter` + `filtfilt` is a low-pass Butterworth filter: it keeps the slow moves and removes the fast daily noise. `filtfilt` runs it forward and backward, so the line is not shifted in time.

In [ ]:
part = df.loc['2020':'2022', 'Close']
plt.figure(figsize=(12, 5))
plt.plot(part.index, part, color='lightgray', label='Close')
for window in [11, 51, 101]:
    smooth = signal.savgol_filter(part, window_length=window, polyorder=2)
    # window must be odd, polyorder 2 = small parabola
    plt.plot(part.index, smooth, label=f'Savgol window={window}')
plt.title('Savitzky-Golay filter with different windows (2020-2022)')
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(part.index, part, color='lightgray', label='Close')
for cutoff in [0.1, 0.03, 0.01]:
    b, a = signal.butter(3, cutoff)
    # order 3, cutoff as a share of the highest frequency (smaller = smoother)
    low = signal.filtfilt(b, a, part)
    plt.plot(part.index, low, label=f'Butterworth cutoff={cutoff}')
plt.title('Butterworth low-pass filter with different cutoffs (2020-2022)')
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.legend()
plt.show()

In [ ]:
# how far each filtered line is from the real price on average
for window in [11, 51, 101]:
    smooth = signal.savgol_filter(part, window, 2)
    print('Savgol', window, 'mean abs diff:', round(np.mean(np.abs(part - smooth)), 2))
for cutoff in [0.1, 0.03, 0.01]:
    b, a = signal.butter(3, cutoff)
    print('Butter', cutoff, 'mean abs diff:', round(np.mean(np.abs(part - signal.filtfilt(b, a, part))), 2))

**What I found:** I tried 3 settings for each filter on 2020-2022.
- **Savgol:** window 11 stays very close to the price (mean difference 1.42 USD), and window 101 is much smoother but further from the price (4.22 USD).
- **Butterworth:** cutoff 0.1 keeps the price shape (1.92 USD), and cutoff 0.01 leaves only the big trend (6.46 USD).
- So there is always a trade-off: more smoothing removes more noise but also more real moves. For me, Savgol 51 and Butterworth 0.03 looked best for seeing the trend. Both filters also have no delay, unlike the rolling mean, which always lags behind the price.

### Statistical Functions in NumPy
First I make a 50-day moving average with `np.convolve` and check it against pandas `rolling`.

In [ ]:
window = 50
kernel = np.ones(window) / window
# 50 weights of 1/50, so the sum is an average
ma_conv = np.convolve(df['Close'].values, kernel, mode='valid')
# 'valid' only keeps positions with a full 50 days
ma_roll = df['MA50'].dropna().values
print(len(ma_conv), len(ma_roll))
print('Biggest difference:', np.max(np.abs(ma_conv - ma_roll)))

In [ ]:
# correlations between price, volume, daily range and returns with np.corrcoef
df['Range'] = df['High'] - df['Low']
metrics = df[['Close', 'Volume', 'Range', 'Return']].dropna()
corr = np.corrcoef(metrics.values.T)
# .T because corrcoef wants one row per variable
print(pd.DataFrame(corr, index=metrics.columns, columns=metrics.columns).round(2))

In [ ]:
# correlation between the 50-day MA of Close and the 50-day MA of Volume, per decade
df['VolMA50'] = df['Volume'].rolling(50).mean()
df['decade'] = df.index.year // 10 * 10
for decade in [1980, 1990, 2000, 2010, 2020]:
    d = df[df['decade'] == decade].dropna(subset=['MA50', 'VolMA50'])
    r = np.corrcoef(d['MA50'], d['VolMA50'])[0, 1]
    print(decade, 'r =', round(r, 2), '| days:', len(d))

In [ ]:
# the same correlation over a moving 2-year window (about 504 trading days)
roll_corr = df['MA50'].rolling(504).corr(df['VolMA50'])
plt.figure(figsize=(12, 4))
plt.plot(roll_corr.index, roll_corr)
plt.axhline(0, color='black', linewidth=0.8)
plt.title('Rolling 2-year correlation: 50-day MA of Close vs 50-day MA of Volume')
plt.xlabel('Date')
plt.ylabel('Correlation')
plt.show()

**What I found:**
- `np.convolve` gives the same 50-day moving average as pandas `rolling`. Both have 10,559 values and the biggest difference is about 6e-14, which is just a rounding error.
- **np.corrcoef between metrics:** Close and the daily High-Low range are strongly correlated (0.87), because a 1% move is more dollars when the price is high. Close and Volume are weakly negative (-0.21). Daily returns have almost no correlation with anything (about 0.00), so the price level or volume does not predict the next day's return.
- **MA of Close vs MA of Volume per decade:** the relationship changes over time. It was positive in the 1980s (0.33), almost none in the 1990s (0.10), and quite positive in the 2000s (0.66), when the price and trading rose together. It was strongly negative in the 2010s (-0.79) and 2020s (-0.76): the price kept rising while fewer shares were traded. The rolling 2-year chart shows the same swings. One reason is the stock splits (2014 and 2020): the volume is not adjusted the same way, so the long-term comparison is not perfect.

## 6. Summary and Insights
- **Data:** 10,608 clean trading days (1981-2023) with no missing values. The only cleaning needed was turning `Date` into a real date.
- **Trend:** Apple grew from about 0.11 USD to a top of 182 USD (split-adjusted). Most growth came after 2005, and 2022 brought a drop, which the 50/200 day moving averages show as a death cross.
- **Summary stats:** over the whole period the mean and median are very far apart, so for prices that grow this much I should look at shorter periods.
- **Hypothesis tests:** the average close is clearly different between years (t-test 2021 vs 2022 p about 4.7e-26, ANOVA 2018-2022 p about 0).
- **Returns:** the daily returns are not normal (kurtosis 18, Shapiro and normaltest p about 0). There are many more extreme days than a normal curve predicts.
- **Signal processing:** Savitzky-Golay and Butterworth filters both smooth the price without the lag of a rolling mean. The window or cutoff controls how much detail stays.
- **NumPy:** `np.convolve` reproduces the rolling mean exactly. `np.corrcoef` shows that the link between the price trend and the volume trend changes from decade to decade, from +0.66 in the 2000s to -0.79 in the 2010s, so one correlation number for 42 years would hide this.

## 7. Reflection
- **Dates:** at first pandas read the dates as text, and 02/01/1981 can mean 2 January or 1 February. I fixed it by giving the format `%d/%m/%Y` to `pd.to_datetime`.
- **Huge price range:** prices from 0.1 to 180 USD made the first chart and the summary statistics hard to read. I solved it with a log-scale chart and by also calculating the stats for 2018-2022 only.
- **Candlestick chart:** I did not want a new library, so I built the candles from `vlines` and `bar`. With all 42 years the candles were invisible, so I used the last 60 trading days.
- **Shapiro-Wilk limit:** with more than 5000 values `shapiro` gives a warning, so I used a random sample of 5000 and checked it with `normaltest` on all the data.
- **Filters:** at first I did not understand the Butterworth cutoff number. Trying 0.1, 0.03 and 0.01 next to each other made it clear: smaller means smoother.
- **t-test assumptions:** I learned that daily prices depend on the day before, so the t-test p-values are not perfect for stock data. Here the differences were big enough that it did not change my conclusion.